In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Data paths
TRAIN_PATH = "../data/in/cattle_data_train.csv"
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID", "Farm_ID", "Feed_Quantity_lb", "Climate_Zone",
    "Management_System", "Feed_Type", "Feeding_Frequency",
    "Walking_Distance_km", "Grazing_Duration_hrs", "Rumination_Time_hrs",
    "Resting_Hours", "Humidity_percent", "BVD_Vaccine", "FMD_Vaccine",
    "Brucellosis_Vaccine", "HS_Vaccine", "BQ_Vaccine", "Housing_Score",
    "Body_Condition_Score", "Milking_Interval_hrs", "Breed",
]

CATEGORICAL_FEATURES = ["Date", "Young", "Lactation_Stage"]

STANDARD_SCALED_FEATURES = [
    "Feed_Quantity_kg", "Water_Intake_L", "Parity",
    "Ambient_Temperature_C", "Previous_Week_Avg_Yield",
    "Days_in_Milk", "Age_Months", "Weight_kg",
]

def preprocess(dtrain, dtest, scaler=None):
    """Preprocess training and test data"""
    # Convert month to season
    def month_to_season(m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    for df in [dtrain, dtest]:
        months = pd.to_datetime(df['Date']).dt.month
        df.drop(columns=['Date'], inplace=True)
        df['Date'] = months.apply(month_to_season)
        df['Young'] = (df['Age_Months'] < 60).astype(int)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median()
    dtrain.loc[dtrain["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val

    # Drop features
    dtrain = dtrain.drop(DROP_FEATURES, axis=1)
    dtest = dtest.drop(DROP_FEATURES, axis=1)

    # One-hot encode
    dtrain = pd.get_dummies(dtrain, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtest = pd.get_dummies(dtest, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtrain, dtest = dtrain.align(dtest, join='left', axis=1, fill_value=0)

    # Standardize
    if scaler is None:
        scaler = StandardScaler()
        dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform(dtrain[STANDARD_SCALED_FEATURES])
    else:
        dtrain[STANDARD_SCALED_FEATURES] = scaler.transform(dtrain[STANDARD_SCALED_FEATURES])
    
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform(dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

# Load data and create validation split
print("Loading data and creating validation split...")
train_data = pd.read_csv(TRAIN_PATH)

# Split: 60% train, 20% dev (for finding optimal iters), 20% validation (for final eval)
X_temp, X_val, y_temp, y_val = train_test_split(
    train_data.drop(TARGET_FEATURE, axis=1),
    train_data[TARGET_FEATURE], 
    test_size=0.2, 
    random_state=42
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, 
    test_size=0.25,  # 0.25 of 80% = 20% of total
    random_state=42
)

print(f"Train size: {len(X_train)}, Dev size: {len(X_dev)}, Validation size: {len(X_val)}")

# Preprocess
X_train_proc, X_dev_proc, scaler = preprocess(X_train.copy(), X_dev.copy())
_, X_val_proc, _ = preprocess(X_train.copy(), X_val.copy(), scaler=scaler)

# Model template
model_template = MLPRegressor(
    hidden_layer_sizes=(110, 110, 110),
    activation="tanh",
    learning_rate_init=0.00003,
    learning_rate="adaptive",
    early_stopping=False,
    n_iter_no_change=20,
    verbose=False,
    warm_start=True,
    max_iter=10,
    random_state=1
)

TOLERANCE = 0.00005

In [ ]:
# Find optimal iterations using train/dev split
print("\n" + "="*60)
print("FINDING OPTIMAL ITERATIONS")
print("="*60)

model = clone(model_template)
prev_rmse = float('inf')
dev_rmse = 0
total_iterations = 0

while dev_rmse + TOLERANCE < prev_rmse:
    if dev_rmse > 0:
        prev_rmse = dev_rmse
    
    model.fit(X_train_proc, y_train)
    
    y_train_pred = model.predict(X_train_proc)
    y_dev_pred = model.predict(X_dev_proc)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    dev_rmse = np.sqrt(mean_squared_error(y_dev, y_dev_pred))
    
    total_iterations += 10
    print(f"Iteration {total_iterations}: Train RMSE={train_rmse:.5f}, Dev RMSE={dev_rmse:.5f}")

print(f"\nOptimal iterations found: {total_iterations}")

# Approach 1: Single model trained on train+dev
print("\n" + "="*60)
print("APPROACH 1: SINGLE MODEL ON ALL TRAINING DATA")
print("="*60)

X_all_train = pd.concat([X_train, X_dev])
y_all_train = pd.concat([y_train, y_dev])

X_all_train_proc, X_val_proc_1, scaler_1 = preprocess(X_all_train.copy(), X_val.copy())

single_model = clone(model_template)
single_model.max_iter = total_iterations
single_model.fit(X_all_train_proc, y_all_train)

y_val_pred_single = single_model.predict(X_val_proc_1)
val_rmse_single = np.sqrt(mean_squared_error(y_val, y_val_pred_single))

print(f"Single Model - Validation RMSE: {val_rmse_single:.5f}")

In [ ]:
# Approach 2: 5-fold ensemble + Full data model
print("\n" + "="*60)
print("APPROACH 2: HYBRID ENSEMBLE (5 K-FOLD + 1 FULL)")
print("="*60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_models = []
fold_scalers = []
fold_iterations = []

X_all_train = pd.concat([X_train, X_dev])
y_all_train = pd.concat([y_train, y_dev])
X_all_train_reset = X_all_train.reset_index(drop=True)
y_all_train_reset = y_all_train.reset_index(drop=True)

# Train K-Fold models
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all_train_reset)):
    print(f"\nTraining fold {fold_idx + 1}/5...")
    
    # Split fold into train and internal validation for early stopping
    X_fold_train = X_all_train_reset.iloc[train_idx]
    y_fold_train = y_all_train_reset.iloc[train_idx]
    X_fold_val = X_all_train_reset.iloc[val_idx]
    y_fold_val = y_all_train_reset.iloc[val_idx]
    
    # Preprocess fold data
    X_fold_train_proc, X_fold_val_proc, fold_scaler = preprocess(
        X_fold_train.copy(), 
        X_fold_val.copy()
    )
    
    # Find optimal iterations for this fold using tolerance method
    fold_model = clone(model_template)
    prev_fold_rmse = float('inf')
    fold_val_rmse = 0
    fold_iters = 0
    
    while fold_val_rmse + TOLERANCE < prev_fold_rmse:
        if fold_val_rmse > 0:
            prev_fold_rmse = fold_val_rmse
        
        fold_model.fit(X_fold_train_proc, y_fold_train)
        
        y_fold_val_pred = fold_model.predict(X_fold_val_proc)
        fold_val_rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_val_pred))
        
        fold_iters += 10
        if fold_iters % 50 == 0:  # Print every 50 iterations
            print(f"  Iteration {fold_iters}: Fold Val RMSE={fold_val_rmse:.5f}")
    
    print(f"  Optimal iterations for fold {fold_idx + 1}: {fold_iters}")
    fold_iterations.append(fold_iters)
    
    fold_models.append(fold_model)
    fold_scalers.append(fold_scaler)
    
    # Evaluate on held-out validation set
    _, X_val_proc_fold, _ = preprocess(
        X_fold_train.copy(), 
        X_val.copy(),
        scaler=fold_scaler
    )
    y_fold_pred = fold_model.predict(X_val_proc_fold)
    fold_rmse = np.sqrt(mean_squared_error(y_val, y_fold_pred))
    print(f"  Fold {fold_idx + 1} on True Validation RMSE: {fold_rmse:.5f}")

# Train full-data model
print("\n" + "="*60)
print("Training full-data model (6th ensemble member)...")
print("="*60)

avg_iters = int(np.mean(fold_iterations))
print(f"Using average iterations from folds: {avg_iters}")

X_all_train_proc, X_val_proc_full, full_scaler = preprocess(
    X_all_train.copy(),
    X_val.copy()
)

full_model = clone(model_template)
full_model.max_iter = avg_iters
full_model.fit(X_all_train_proc, y_all_train)

y_val_pred_full = full_model.predict(X_val_proc_full)
full_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_full))
print(f"Full model - Validation RMSE: {full_rmse:.5f}")

# Add full model to ensemble
fold_models.append(full_model)
fold_scalers.append(full_scaler)

# Ensemble predictions
print("\n" + "="*60)
print("Generating ensemble predictions...")
print("="*60)

# First get 5-fold ensemble (without full model)
ensemble_preds_5fold = []
for idx in range(5):  # First 5 models only
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    _, X_val_proc_fold, _ = preprocess(
        X_all_train.copy(), 
        X_val.copy(), 
        scaler=fold_scaler
    )
    fold_pred = fold_model.predict(X_val_proc_fold)
    ensemble_preds_5fold.append(fold_pred)
    print(f"Fold {idx + 1} model prediction mean: {fold_pred.mean():.5f}")

y_val_pred_5fold = np.mean(ensemble_preds_5fold, axis=0)
val_rmse_5fold = np.sqrt(mean_squared_error(y_val, y_val_pred_5fold))

print(f"\n5-Fold Ensemble - Validation RMSE: {val_rmse_5fold:.5f}")

# Now add full model for 6-model ensemble
ensemble_preds_6model = ensemble_preds_5fold.copy()

_, X_val_proc_fold, _ = preprocess(
    X_all_train.copy(), 
    X_val.copy(), 
    scaler=fold_scalers[-1]  # Full model scaler
)
full_pred = fold_models[-1].predict(X_val_proc_fold)
ensemble_preds_6model.append(full_pred)
print(f"Full model prediction mean: {full_pred.mean():.5f}")

y_val_pred_6model = np.mean(ensemble_preds_6model, axis=0)
val_rmse_6model = np.sqrt(mean_squared_error(y_val, y_val_pred_6model))

# Summary
print("\n" + "="*60)
print("ENSEMBLE COMPARISON")
print("="*60)
print(f"Full model alone:        {full_rmse:.5f}")
print(f"5-Fold Ensemble:         {val_rmse_5fold:.5f}")
print(f"Hybrid (6-model) Ensemble: {val_rmse_6model:.5f}")
print(f"\nImprovement (5-fold vs full):  {full_rmse - val_rmse_5fold:.5f}")
print(f"Improvement (6-model vs full): {full_rmse - val_rmse_6model:.5f}")
print(f"Improvement (6-model vs 5-fold): {val_rmse_5fold - val_rmse_6model:.5f}")